## Упражнение 03. Агрегации

In [1]:
import pandas as pd
import sqlite3

### 1. Создаем соединение с базой данных с помощью библиотеки sqlite3

In [2]:
conn=sqlite3.connect('../data/checking-logs.sqlite')

### 2. Получил схему таблицы test

In [3]:
pd.io.sql.read_sql('PRAGMA table_info(test);',conn)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


### 3. Получил первые десять строк таблицы test, чтобы посмотреть её структуру.

In [4]:
pd.io.sql.read_sql('select* from test limit 10', conn)

,uid,labname,first_commit_ts,first_view_ts
0,user_17,project1,2020-04-18 07:56:45.408648,2020-04-18 10:56:55.833899
1,user_30,laba04,2020-04-18 13:36:53.971502,2020-04-17 22:46:26.785035
2,user_30,laba04s,2020-04-18 14:51:37.498399,2020-04-17 22:46:26.785035
3,user_14,laba04,2020-04-18 15:14:00.312338,2020-04-18 10:53:52.623447
4,user_14,laba04s,2020-04-18 22:30:30.247628,2020-04-18 10:53:52.623447
5,user_19,laba04,2020-04-20 19:05:01.297780,2020-04-21 20:30:38.034966
6,user_25,laba04,2020-04-20 19:16:50.673054,2020-05-09 23:54:54.260791
7,user_21,laba04,2020-04-21 17:48:00.487806,2020-04-22 22:40:36.824081
8,user_30,project1,2020-04-22 12:36:24.053518,2020-04-17 22:46:26.785035
9,user_21,laba04s,2020-04-22 20:09:21.857747,2020-04-22 22:40:36.824081


### 4. Найдем минимальное значение дельты между первым коммитом и дедлайном соответствующей лабораторной для всех пользователей одним запросом.
#### - Выполни объединение с таблицей deadlines;
#### - Разницу отобрази в часах;
#### - Не учитывай лабораторную project1 (у неё более длинные дедлайны - это выброс);
#### - Сохрани значение в DataFrame df_min вместе с соответствующим uid.

In [5]:
query="""
select test.uid as uid,
MIN((unixepoch(test.first_commit_ts) - deadlines.deadlines) / 3600) AS min_delt FROM test
INNER JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1';
"""
df_min=pd.io.sql.read_sql(query,conn)
df_min

,uid,min_delt
0,user_30,-202


### 5. Аналогично найдем максимум (тоже одним запросом). Имя DataFrame - df_max.


In [6]:
query= """
select test.uid AS uid,
MAX((unixepoch(test.first_commit_ts) - deadlines.deadlines) / 3600) AS max_delt FROM test
INNER JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1';
"""
df_max= pd.io.sql.read_sql(query,conn)
df_max

,uid,max_delt
0,user_25,-2


### 6. Аналогично найдем среднее (тоже одним запросом). На этот раз в DataFrame не должно быть колонки uid. Имя DataFrame - df_avg.

In [7]:
sql_query: str = """
SELECT
AVG((unixepoch(test.first_commit_ts) - deadlines.deadlines) / 3600) AS avg_delt FROM test
INNER JOIN deadlines ON test.labname = deadlines.labs
WHERE test.labname != 'project1';
"""
df_avg=pd.io.sql.read_sql(sql_query,conn)
df_avg

,avg_delt
0,-89.125


### 7. Протестируем гипотезу, что у пользователей, которые посещали ленту новостей несколько раз, дельта между первым коммитом и дедлайном ниже. Для этого посчитай коэффициент корреляции между числом просмотров ленты и дельтой.
### -Одним запросом создай таблицу с колонками: "uid", "avg_diff", "pageviews";
### -"uid" - идентификаторы, существующие в test;
### -"avg_diff" - средняя дельта между первым коммитом и дедлайном по пользователю;
### -"pageviews" - число посещений ленты новостей по пользователю;
### -Не учитывай project1;
### -Сохрани результат в DataFrame views_diff;
### -Используй метод Pandas corr() для вычисления коэффициента корреляции между числом просмотров и дельтой.

In [8]:
sql_query = """
SELECT test.uid as uid, 
AVG((unixepoch(test.first_commit_ts) - deadlines.deadlines) / 3600) AS avg_diff, 
COUNT(pageviews.datetime) AS pageviews
FROM test
INNER JOIN deadlines ON test.labname = deadlines.labs
LEFT OUTER JOIN pageviews ON test.uid = pageviews.uid
WHERE test.labname != 'project1'
GROUP BY test.uid;
"""
views_diff = pd.io.sql.read_sql(sql_query, conn)

result = float(round(views_diff["pageviews"].corr(views_diff["avg_diff"]), 4))
print(result)


-0.1858


### 8. Закрываем соединение

In [9]:
conn.close()